# 05 — Modelling dataset

Joins prices, ENTSO-E fundamentals and gas into **one hourly table**, aligned on `ts_utc`, that every model reads.

- **Input:** the processed parquet files of 01, 03 and 04
- **Output:** `data/processed/dataset_hourly_2021_2025q3.parquet` (41,615 × 14, zero `NaN`)
- **Features:** load forecast, solar and wind forecast, **residual load** (load − wind − solar), gas price D-1.
- **Key rule:** a variable is a valid feature only if its value was known at **12:00 of D-1**, the day-ahead gate closure.

In [1]:
# --- Setup: load the three processed files ---------------------------------
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
PROCESSED = PROJECT_ROOT / "data" / "processed"

prices = pd.read_parquet(PROCESSED / "mgp_prices_hourly_2021_2025q3.parquet")
entsoe = pd.read_parquet(PROCESSED / "entsoe_hourly_2020_2025.parquet")
gas = pd.read_parquet(PROCESSED / "gas_mgp_daily_2020_2025.parquet")

# Time column of each frame: needed to compare ranges and units before joining
TIME_COLUMNS = {"prices": "ts_utc", "entsoe": "ts_utc", "gas": "gas_day"}
frames = {"prices": prices, "entsoe": entsoe, "gas": gas}

for name, df in frames.items():
    time_col = TIME_COLUMNS[name]
    print("=" * 72)
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} cols")
    # The dtype shows both the unit (ns/us) and the timezone: they must match
    # across frames before any merge on ts_utc
    print(f"time column '{time_col}': {df[time_col].dtype}")
    print(f"range: {df[time_col].min()} -> {df[time_col].max()}")
    print(df.dtypes.to_string())

prices: 41615 rows x 26 cols
time column 'ts_utc': datetime64[us, UTC]
range: 2020-12-31 23:00:00+00:00 -> 2025-09-30 21:00:00+00:00
date                                    int64
hour                                    int64
national_price                        float64
AUST                                  float64
BSP                                   float64
CALA                                  float64
CNOR                                  float64
COAC                                  float64
CORS                                  float64
CSUD                                  float64
FRAN                                  float64
GREC                                  float64
MALT                                  float64
MONT                                  float64
NORD                                  float64
SARD                                  float64
SICI                                  float64
SLOV                                  float64
SUD                                   f

In [2]:
# --- Skeleton: the target defines which rows exist -------------------------
TARGET = "national_price"

dataset = prices[["ts_utc", "ts_local", TARGET]].copy()

# --- Join ENTSO-E fundamentals on ts_utc -----------------------------------
# how="left": the price grid is the spine, ENTSO-E hours outside the
# perimeter (2020, Oct-Dec 2025) are simply not brought in.
# validate="one_to_one": raises if either side has duplicated timestamps.
# ts_local is dropped from the right side, otherwise pandas would create
# ts_local_x / ts_local_y.
dataset = dataset.merge(
    entsoe.drop(columns=["ts_local"]),
    on="ts_utc",
    how="left",
    validate="one_to_one",
    indicator=True,
)

# Every price hour must have found its ENTSO-E row: the grids were built
# independently (GME Ora 1-25 vs ENTSO-E positions), so this is a real test
print(dataset["_merge"].value_counts().to_string())
assert (dataset["_merge"] == "both").all(), "some price hours have no ENTSO-E row"
dataset = dataset.drop(columns="_merge")

print(dataset.shape)
print("\nNaN inside the perimeter:")
print(dataset.isna().sum().to_string())

_merge
both          41615
left_only         0
right_only        0
(41615, 11)

NaN inside the perimeter:
ts_utc                          0
ts_local                        0
national_price                  0
load_forecast_mw                1
solar_forecast_mw             243
wind_onshore_forecast_mw      267
load_actual_mw                  0
solar_actual_mw                 0
wind_onshore_actual_mw          0
wind_offshore_actual_mw     12263
gas_actual_mw                   0


In [4]:
# --- Fallback for missing TSO forecasts ------------------------------------
FORECAST_COLUMNS = ["load_forecast_mw", "solar_forecast_mw", "wind_onshore_forecast_mw"]

# Work on the full ENTSO-E frame (it starts in 2020), not on dataset: a gap
# close to the start of the perimeter can still find its previous days
forecasts = entsoe.set_index("ts_utc")[FORECAST_COLUMNS]

# shift(24) means "24 hours earlier" only if the grid has no holes in rows
steps = forecasts.index.to_series().diff().dropna()
assert (steps == pd.Timedelta(hours=1)).all(), "ENTSO-E grid is not contiguous"

# Flag BEFORE filling: this is what lets 06_baseline decide later whether to
# train/evaluate on imputed hours or to drop them (option a stays available)
imputed = forecasts.isna().any(axis=1)

filled = forecasts.copy()
for days_back in range(1, 8):
    # Same UTC hour, days_back days earlier. The previous day's forecast was
    # published two days before delivery, so it is known at 12:00 D-1: no
    # leakage. UTC rather than local time on purpose: solar follows the sun,
    # and solar noon does not jump by one hour at DST like the local clock.
    # Always shift the ORIGINAL forecasts: never propagate an imputed value.
    filled = filled.fillna(forecasts.shift(24 * days_back))

# Bring the filled columns and the flag into the dataset, aligned on ts_utc
dataset[FORECAST_COLUMNS] = filled.reindex(dataset["ts_utc"]).to_numpy()
dataset["forecast_imputed"] = imputed.reindex(dataset["ts_utc"]).to_numpy()

# Check ONLY inside the perimeter: outside it there are long gaps we do not
# need to fill (the onshore wind forecast only starts on 14/12/2020)
assert dataset[FORECAST_COLUMNS].notna().all().all(), "perimeter gap longer than 7 days"

# --- Offshore wind: zero before the plant existed --------------------------
# Not an imputation: before its first published value the plant was not
# producing, so 0 MW is the true value. Only the leading block is filled;
# a NaN AFTER the first valid value would be a real gap and must stay visible.
offshore = entsoe.set_index("ts_utc")["wind_offshore_actual_mw"]
before_plant = dataset["ts_utc"] < offshore.first_valid_index()
dataset.loc[before_plant, "wind_offshore_actual_mw"] = 0.0

print("imputed hours in perimeter:", int(dataset["forecast_imputed"].sum()))
print("imputed days:", dataset.loc[dataset["forecast_imputed"], "ts_local"].dt.date.nunique())
print("\nNaN left:")
print(dataset.isna().sum().to_string())
assert dataset.isna().sum().sum() == 0

imputed hours in perimeter: 267
imputed days: 12

NaN left:
ts_utc                      0
ts_local                    0
national_price              0
load_forecast_mw            0
solar_forecast_mw           0
wind_onshore_forecast_mw    0
load_actual_mw              0
solar_actual_mw             0
wind_onshore_actual_mw      0
wind_offshore_actual_mw     0
gas_actual_mw               0
forecast_imputed            0


In [5]:
# --- Residual load: the demand that dispatchable plants must cover ---------
# Built ONLY from forecasts, so it is known before the MGP gate closure.
# Offshore wind is not included: ENTSO-E publishes no forecast for it, and its
# output (a few tens of MW) is negligible against a national load of 20-50 GW.
dataset["residual_load_mw"] = (
    dataset["load_forecast_mw"]
    - dataset["solar_forecast_mw"]
    - dataset["wind_onshore_forecast_mw"]
)

# Yearly picture: growing solar should push the minimum down year after year
summary = (
    dataset.groupby(dataset["ts_local"].dt.year)["residual_load_mw"]
           .describe()[["mean", "min", "50%", "max"]]
           .round(0)
)
print(summary.to_string())

             mean     min      50%      max
ts_local                                   
2021      28736.0  6817.0  28382.0  47708.0
2022      28409.0  3765.0  28096.0  46817.0
2023      26862.0  6024.0  26549.0  46670.0
2024      26610.0  4893.0  26370.0  45644.0
2025      25251.0 -2038.0  25072.0  46255.0


In [6]:
# --- Low tail of the residual load: trend and suspicious hours -------------
year = dataset["ts_local"].dt.year

# Low quantiles instead of the minimum: a single hour is dominated by weather
# and holidays, the 1% / 5% quantiles show the structural trend
low_tail = (
    dataset.groupby(year)["residual_load_mw"]
           .quantile([0.01, 0.05])
           .unstack()
           .round(0)
)
low_tail["hours_below_zero"] = (dataset["residual_load_mw"] < 0).groupby(year).sum()
print(low_tail.to_string())

# The 10 lowest hours, next to the price and to the residual load computed
# from ACTUALS: if forecast and actual disagree wildly, or the price is
# normal, the forecast value is suspicious
lowest = (
    dataset.assign(
        residual_load_actual_mw=lambda d: d["load_actual_mw"]
        - d["solar_actual_mw"]
        - d["wind_onshore_actual_mw"]
        - d["wind_offshore_actual_mw"]
    )
    .nsmallest(10, "residual_load_mw")
    [["ts_local", "national_price", "load_forecast_mw", "solar_forecast_mw",
      "wind_onshore_forecast_mw", "residual_load_mw", "residual_load_actual_mw",
      "forecast_imputed"]]
)
print()
print(lowest.to_string(index=False))

             0.01     0.05  hours_below_zero
ts_local                                    
2021      14536.0  17915.0                 0
2022      14133.0  17784.0                 0
2023      12551.0  16454.0                 0
2024       9994.0  15440.0                 0
2025       6115.0  11838.0                 9

                 ts_local  national_price  load_forecast_mw  solar_forecast_mw  wind_onshore_forecast_mw  residual_load_mw  residual_load_actual_mw  forecast_imputed
2025-05-25 14:00:00+02:00            0.00          21614.75           16404.75                   7248.00          -2038.00                  5355.00             False
2025-05-25 13:00:00+02:00            0.00          22490.50           16962.25                   7171.75          -1643.50                  4241.25             False
2025-05-25 15:00:00+02:00            0.00          21317.50           14983.50                   7301.50           -967.50                  7106.00             False
2025-05-25 12:00:00+

In [7]:
# --- Gas price feature: last gas day fully known at the MGP gate closure ----
# The auction for delivery day D closes at 12:00 of D-1 for ALL 24 hours at
# once, so every hour of D gets the same gas information: the price of gas
# day D-1, final at 02:30 of D-1. The 06:00 gas-day boundary does not matter
# here: it matters for the PHYSICAL fuel cost of an hour, not for what an
# operator knows when bidding.

# Electricity delivery date = local calendar day of the hour, as a naive
# midnight timestamp so that it has the same dtype as gas_day.
# normalize() BEFORE dropping the timezone: midnight always exists in Rome
# (the DST switch happens at 02:00/03:00), so nothing is shifted.
delivery_date = dataset["ts_local"].dt.normalize().dt.tz_localize(None)
gas_day_used = delivery_date - pd.Timedelta(days=1)

# reindex needs a unique index on the lookup side: one row per gas day
gas_price = gas.set_index("gas_day")["price_eur_mwh"]
assert gas_price.index.is_unique

dataset["gas_price_d1_eur_mwh"] = gas_price.reindex(gas_day_used).to_numpy()

# --- Checks ------------------------------------------------------------------
assert dataset["gas_price_d1_eur_mwh"].notna().all(), "gas day missing for some hours"

# One and only one gas value per delivery day, whatever its number of hours
per_day = dataset.groupby(delivery_date)["gas_price_d1_eur_mwh"].nunique()
assert (per_day == 1).all(), "more than one gas price within a delivery day"

# First delivery day of the perimeter must use the gas day before it (2020-12-31)
first = pd.Timestamp("2020-12-31")
assert dataset["gas_price_d1_eur_mwh"].iloc[0] == gas_price.loc[first]

sample = dataset.assign(gas_day_used=gas_day_used)
print(sample.loc[sample["ts_local"].dt.hour.isin([0, 5, 6, 23]),
                 ["ts_local", "gas_day_used", "gas_price_d1_eur_mwh"]].head(8).to_string(index=False))

                 ts_local gas_day_used  gas_price_d1_eur_mwh
2021-01-01 00:00:00+01:00   2020-12-31                17.544
2021-01-01 05:00:00+01:00   2020-12-31                17.544
2021-01-01 06:00:00+01:00   2020-12-31                17.544
2021-01-01 23:00:00+01:00   2020-12-31                17.544
2021-01-02 00:00:00+01:00   2021-01-01                16.868
2021-01-02 05:00:00+01:00   2021-01-01                16.868
2021-01-02 06:00:00+01:00   2021-01-01                16.868
2021-01-02 23:00:00+01:00   2021-01-01                16.868


In [8]:
# --- Column roles and order ------------------------------------------------
# Written down explicitly so that 06_baseline can import the lists instead of
# guessing: anything in DIAGNOSTICS must NEVER enter a model.
TIME_COLUMNS = ["ts_utc", "ts_local"]
TARGET = "national_price"
FEATURES = [
    "load_forecast_mw",
    "solar_forecast_mw",
    "wind_onshore_forecast_mw",
    "residual_load_mw",
    "gas_price_d1_eur_mwh",
]
FLAGS = ["forecast_imputed"]
DIAGNOSTICS = [                       # not known at 12:00 D-1
    "load_actual_mw",
    "solar_actual_mw",
    "wind_onshore_actual_mw",
    "wind_offshore_actual_mw",
    "gas_actual_mw",
]
ORDERED = TIME_COLUMNS + [TARGET] + FEATURES + FLAGS + DIAGNOSTICS

# No column lost or forgotten along the way
assert set(ORDERED) == set(dataset.columns), set(dataset.columns) ^ set(ORDERED)
dataset = dataset[ORDERED].sort_values("ts_utc").reset_index(drop=True)

# --- Final checks ----------------------------------------------------------
assert len(dataset) == 41615, len(dataset)
assert dataset["ts_utc"].is_unique and dataset["ts_utc"].is_monotonic_increasing
assert dataset.isna().sum().sum() == 0

# --- Persist and verify the round-trip -------------------------------------
out_path = PROCESSED / "dataset_hourly_2021_2025q3.parquet"
dataset.to_parquet(out_path, index=False)

check = pd.read_parquet(out_path)
pd.testing.assert_frame_equal(check, dataset)

print(f"saved {check.shape[0]} rows x {check.shape[1]} cols, "
      f"{out_path.stat().st_size / 1e6:.1f} MB -> {out_path.name}")
print(check.head(3).T.to_string())

saved 41615 rows x 14 cols, 2.2 MB -> dataset_hourly_2021_2025q3.parquet
                                                  0                          1                          2
ts_utc                    2020-12-31 23:00:00+00:00  2021-01-01 00:00:00+00:00  2021-01-01 01:00:00+00:00
ts_local                  2021-01-01 00:00:00+01:00  2021-01-01 01:00:00+01:00  2021-01-01 02:00:00+01:00
national_price                                50.87                      48.19                   44.67966
load_forecast_mw                            22603.0                    22004.0                    20597.0
solar_forecast_mw                               0.0                        0.0                        0.0
wind_onshore_forecast_mw                     2559.0                     2548.0                     2523.0
residual_load_mw                            20044.0                    19456.0                    18074.0
gas_price_d1_eur_mwh                         17.544                     17.544 